# Test du machine Learning, FEATURE ENGINEERING
Le travaille de feature enginnering consiste à transformer les données en variables exploitables par les algorithmes de ML et optimisez l'apprentissage. 
Notre travaille dans cette partie serais de pouvoir : 
- Encoder les données textuelle. (One-Hot Encoding ou le Label encoding) 
-  Gestion des valeurs NaN (Par zéro, la médiane, ou la suppression )
- Créer des variables qui pourront nous etre utiles pour le ML et plus facile
- Standardise les données ou Normaliser 
- Faire la selection des variable qui seront plus utile pour notre modeles


## Faire Un Pipeline
Il est important de faire dans un pipeline pour éviter les data-leakage (fuite de données), facilité la reproductibilité, maintenabilité, Déploiement  

#### Import des données ML et l'utilisation des features

In [4]:
import numpy as np 
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold

In [19]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    # Séparation des données propres au pli actuel
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Étape essentielle : On ajuste le pipeline UNIQUEMENT sur le train du pli
    X_tr_transformed = preprocessor.fit_transform(X_tr)
    
    # Entraînement du modèle pour ce pli
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_tr_transformed, y_tr)
    
    # Sauvegarde des poids calculés sur ce pli
    all_importances.append(rf_model.feature_importances_)
    print(f"Pli {fold + 1}/5 terminé.")

# 3. Calcul de la moyenne des poids sur l'ensemble des 5 plis
mean_importances = np.mean(all_importances, axis=0)

# 4. Récupération des vrais noms des colonnes générées par le pipeline
# On extrait les noms créés par le OneHotEncoder et les autres transformations
feature_names = (
    [c for c in cols_to_zero if c in X.columns] +
    [c for c in cols_to_median if c in X.columns] +
    list(preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out()) +
    [c for c in X.columns if c not in cols_to_zero + cols_to_median + categorical_cols] # colonnes passthrough (ex: age)
)

# 5. Création du DataFrame final des importances
df_importance = pd.DataFrame({
    'Variable': feature_names,
    'Poids Moyen': mean_importances
}).sort_values(by='Poids Moyen', ascending=False)

# 6. Visualisation graphique
plt.figure(figsize=(12, 8))
sns.barplot(x='Poids Moyen', y='Variable', data=df_importance.head(15), palette='viridis')
plt.title('Top 15 des variables les plus importantes (Moyenne Stratified K-Fold)', fontsize=14)
plt.xlabel('Poids (Importance Moyenne)')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

# Affichage du tableau des scores
print("\nClassement des variables les plus importantes :")
print(df_importance.head(15))

NameError: name 'X_train' is not defined

#### Récupération des données 

In [13]:
df_finascore = pd.read_csv("../data/processed/finascore_clean.csv", low_memory=True)

df_finascore.duplicated().sum()
df_finascore.head(5)

,applicant_id,age,pays_x,secteur_activite,revenu_mensuel_xaf,anciennete_emploi,ratio_endettement,historique_credit,nb_credits_actifs,mobile_money_score,...,pays_y,seuil_score,volume_mensuel,jours_depuis_dernier_credit,nb_transactions_mois,volume_entrant_xaf,volume_sortant_xaf,regularite_score,operateur,anciennete_compte_mois
0,FIN-1027-1732,39,CMR,Commerce,239600.0,37.0,0.149,NaN,0,63.4,...,NaN,NaN,NaN,NaN,39.0,175700.0,89300.0,57.5,MTN,14.0
1,FIN-1001-1931,47,GAB,Agriculture,288300.0,96.0,0.284,NaN,0,82.8,...,NaN,NaN,NaN,NaN,58.0,48200.0,42900.0,70.1,MTN,61.0
2,FIN-1016-1847,39,CMR,Agriculture,38000.0,105.0,0.669,632.0,3,58.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FIN-1007-1279,56,COG,Commerce,56800.0,120.0,0.096,NaN,2,61.7,...,NaN,NaN,NaN,NaN,47.0,108700.0,69200.0,77.1,MIXTE,11.0
4,FIN-1032-1614,63,CMR,Pastoral,73100.0,83.0,0.499,306.0,2,NaN,...,CMR,523.0,61.0,10.0,43.0,134700.0,110700.0,85.1,MTN,1.0


In [ ]:
# Présence ou absence d'historique 
df_finascore['flag_primo_demandeur'] = df_finascore['historique_credit'].isna()

# Voir si le client ne depense plus qu'il ne gagne (pour la capacité réel à rembourser et normaliser la comparaison des montants)
df_finascore['solde_flux_mm'] = df_finascore['volume_entrant_xaf'] - df_finascore['volume_sortant_xaf']
df_finascore['ratio_flux_mm'] =  df_finascore["volume_entrant_xaf"]/df_finascore["volume_sortant_xaf"]

# Intensité de ses depenses et revenue 
df_finascore["volume_mm_total"] = df_finascore["volume_entrant_xaf"] + df_finascore["volume_sortant_xaf"] 
df_finascore["ratio_volume_mm_revenu"] =  df_finascore["volume_mm_total"]/ df_finascore["revenu_mensuel_xaf"]

# Fiabilité du score sur le temps, la regularité de son argent sur le temps
df_finascore["score_regularite_pondere"] = df_finascore["regularite_score"] * np.log1p(df_finascore["anciennete_compte_mois"])

# Savoir son taux de retard pour les crédits
df_finascore["taux_retard_credit"] = df_finascore["total_retards"]/df_finascore["nb_credit"]

# Permet de savoir si le client est risqué 
df_finascore["montant_moyen_credit_sur_revenu"] = df_finascore["avg_credit_xaf"]/df_finascore["revenu_mensuel_xaf"]
df_finascore["montant_total_credit_sur_revenu"] = df_finascore["total_montant_xaf"]/df_finascore["revenu_mensuel_xaf"]

# réduit l'effet des très hauts revenus
df["log_revenu_mensuel_xaf"] = np.log1p(df["revenu_mensuel_xaf"])

# Plus le dernier crédit est ancien, moins l'information sur le client est récente.
df_finascore["anciennete_credit_normalisee"] = 1 / (
        1 + df_finascore["jours_depuis_dernier_credit"]
    )

# Si aucun crédit précédent, cette variable doit rester NaN.
df_finascore.loc[
        df_finascore["jours_depuis_dernier_credit"].isna(),
        "anciennete_credit_normalisee"
    ] = np.nan

# Feature qui permet de mettre directement un default
df_finascore["flag_surendette"] = (
        df_finascore["ratio_endettement"] > 1
    ).astype(int)





#### Création des variables utiles

In [ ]:
df_finascore['flag_primo_demandeur']

#### Définir la cible et diviser les valeurs categorielle et numérique

In [ ]:
target = 'defaut_paiement'

## Division des données X,y
X = df_finascore(columns=[target]  + ['applicant_id', 'date_demande'], errors='ignore')
y = df_finascore[target]

# Train test split Séparation donnée d'entrainement et de test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y) # Assurer la reproductibilité 

TypeError: 'DataFrame' object is not callable

#### Encodage
Encoder les valeurs textuelle, pour les rendres utilisable par le modèle. 

In [ ]:
## Valeurs categoriel
## Division des valeurs categoriel et des valeurs numériques
categorical = ['secteur_activite', 'zone', 'pays', 'operateur', 'statut_final']

df_encoded = pd.get_dummies(df_finascore, columns=categorical, drop_first=True)
# numerical = df_finascore.drop[categorical]

,applicant_id,age,revenu_mensuel_xaf,anciennete_emploi,ratio_endettement,historique_credit,nb_credits_actifs,mobile_money_score,date_demande,defaut_paiement,total_montant_xaf,avg_credit_xaf,total_retards,max_retard,nb_credit,jours_depuis_dernier_credit,nb_transactions_mois,volume_entrant_xaf,volume_sortant_xaf,regularite_score,anciennete_compte_mois,secteur_activite_Artisanat,secteur_activite_Commerce,secteur_activite_Elevage,secteur_activite_Pastoral,secteur_activite_Service,zone_Peri-urbain,zone_Rural,zone_Semi-urbain,zone_Urbain,pays_CMR,pays_COG,pays_GAB,pays_GNQ,pays_RCA,pays_TCD,operateur_MIXTE,operateur_MTN,operateur_Mixte,operateur_ORANGE,statut_final_En_cours,statut_final_Rembourse,statut_final_Restructure
0,FIN-1004-1207,34,203800.0,104.0,0.290,459.0,3,NaN,2022-05-28,0,NaN,NaN,NaN,NaN,NaN,NaN,44.0,175400.0,109100.0,93.2,65.0,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,True,False,False,False,False,False
1,FIN-1006-1583,61,195000.0,0.0,0.385,673.0,2,47.5,2024-09-25,0,810000.0,2.700000e+05,11.0,99.0,3.0,1393.0,4.0,76200.0,77200.0,26.6,83.0,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True
2,FIN-1013-1538,48,94600.0,3.0,0.040,NaN,0,75.7,2025-04-07,0,1410000.0,4.700000e+05,0.0,0.0,3.0,1765.0,53.0,17100.0,15000.0,69.2,61.0,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False
3,FIN-1038-1447,49,311400.0,34.0,0.296,NaN,1,51.8,2023-10-17,0,535000.0,2.675000e+05,0.0,0.0,2.0,463.0,NaN,NaN,NaN,NaN,NaN,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False
4,FIN-1022-1267,26,704400.0,0.0,0.216,529.0,3,NaN,2024-08-30,1,430000.0,2.150000e+05,0.0,0.0,2.0,1065.0,11.0,153000.0,89000.0,50.7,2.0,False,False,False,False,True,False,False,False,True,True,False,False,False,False,False,False,False,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50995,FIN-1011-1284,33,192900.0,73.0,0.432,366.0,2,77.8,2025-12-14,0,800000.0,2.000000e+05,0.0,0.0,4.0,938.0,21.0,29800.0,22100.0,48.7,33.0,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,False
50996,FIN-1044-1732,50,2077500.0,0.0,0.243,NaN,2,12.8,2024-09-28,0,NaN,NaN,NaN,NaN,NaN,NaN,14.0,52000.0,41600.0,81.1,21.0,False,False,False,False,True,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False
50997,FIN-1038-1158,40,37300.0,101.0,0.422,NaN,2,65.3,2022-10-19,0,1380000.0,6.900000e+05,0.0,0.0,2.0,1154.0,NaN,NaN,NaN,NaN,NaN,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False
50998,FIN-1000-1860,60,616500.0,72.0,0.332,NaN,0,75.2,2024-10-30,0,1560000.0,5.200000e+05,0.0,0.0,3.0,784.0,18.0,79900.0,66900.0,60.5,52.0,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False
